In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import pandas as pd
import numpy as np
import cv2
import os


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split 

In [ ]:
csv_file = 'english.csv'
data = pd.read_csv(csv_file)

In [ ]:
from PIL import Image

def load_and_preprocess_image(Img, target_size=(128, 128)):
    with Image.open(Img) as img:
        img = img.resize(target_size)
        image = np.array(img) / 255.0
    return image

In [ ]:
images = []
labels = []
for index, row in data.iterrows():
    image_path = row['image']
    label = row['label']
    image = load_and_preprocess_image(image_path)
    images.append(image)
    labels.append(label)  

In [ ]:
images = np.array(images, dtype=np.float32)

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)
labels = np.array(labels, dtype=np.int32)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(label_encoder.classes_), activation='softmax')
])

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(x_train, y_train, epochs=15, validation_data=(x_test, y_test), batch_size=32)

loss, accuracy = model.evaluate(x_test, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

In [ ]:
test_image_path = 'a.png'
test_image = load_and_preprocess_image(test_image_path)
test_image = np.expand_dims(test_image, axis=0)  

In [ ]:
predictions = model.predict(test_image)
predicted_class = np.argmax(predictions, axis=1)

In [ ]:
decoded_class = label_encoder.inverse_transform(predicted_class)
print(f'Predicted class: {decoded_class[0]}')